# Preliminaries

In [2]:
# pip install pytz pyarrow

In [3]:
# GLOBAL
import warnings

# DATA LOADING
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# DATA VISUALIZATION
import matplotlib.pyplot as plt

In [4]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

In [5]:
def max_jobs_in_window(df, window_minutes=5):
    times = pd.to_datetime(df['Submit Time'], unit='s').sort_values().to_numpy()

    window = pd.Timedelta(minutes=window_minutes)
    max_jobs = 0
    left = 0

    for right in range(len(times)):
        while times[right] - times[left] > window:
            left += 1
        max_jobs = max(max_jobs, right - left + 1)

    return max_jobs

# PM100

## Data Loading

In [6]:
df = pd.read_parquet('workloads/pm100/job_table.parquet', engine='pyarrow')

In [7]:
df.columns

Index(['cores_alloc_layout', 'cores_allocated', 'cores_per_task', 'derived_ec',
       'eligible_time', 'end_time', 'group_id', 'job_id', 'job_state', 'nodes',
       'num_cores_req', 'num_cores_alloc', 'num_nodes_req', 'num_nodes_alloc',
       'num_tasks', 'partition', 'priority', 'qos', 'req_nodes', 'req_switch',
       'run_time', 'shared', 'start_time', 'state_reason', 'submit_time',
       'threads_per_core', 'time_limit', 'num_gpus_req', 'num_gpus_alloc',
       'mem_req', 'mem_alloc', 'user_id', 'node_power_consumption',
       'mem_power_consumption', 'cpu_power_consumption'],
      dtype='object')

In [8]:
df.shape

(231238, 35)

In [9]:
df.qos.describe()

count     231238
unique         8
top            1
freq      209600
Name: qos, dtype: object

In [10]:
df.qos.unique()

array(['1', '8', '4', '2', '3', '11', '9', '7'], dtype=object)

## Data Preparation

In [11]:
df['run_time'] = df['run_time'].replace({0: 1})

In [12]:
df = df[df.run_time<df.time_limit*60]

In [13]:
df.qos = df.qos.replace({'normal': 1, 'qos_lowprio': 11, 'm100_qos_bprod': 4, 'm100_qos_dbg': 8})

In [14]:
df['status'] = df['job_state'].replace({'COMPLETED': 1, 'FAILED': 0, 'CANCELLED': 5, 'OUT_OF_MEMORY': -1, 'NODE_FAIL': -1, 'TIMEOUT': -1})

In [15]:
df['wait_time'] = ((df['start_time'] - df['submit_time']).dt.total_seconds()).astype(int)

In [16]:
mb_in_gb = 1024
df['tot_mem_req'] = (df['mem_req'] * df['num_cores_req']) / mb_in_gb

In [17]:
df['time_limit'] = df['time_limit']*60

In [18]:
ts = df.submit_time.min()
init_ts = int(ts.timestamp()) # Convert to Unix timestamp
init_ts

1588729365

In [19]:
# Convert initial timestamp to tz-aware datetime
initial_time = datetime.utcfromtimestamp(init_ts).replace(tzinfo=pytz.UTC)

# Calculate the difference in seconds:
df['submit_time_sec'] = ((df['submit_time'] - initial_time).dt.total_seconds()).astype(int)

In [20]:
required_columns = ['job_id', 'submit_time_sec', 'wait_time', 'run_time', 'num_nodes_alloc', 'run_time', 'mem_alloc', 'num_nodes_req', 'time_limit', 'mem_req', 'status', 'user_id', 'group_id']

In [21]:
swf = df[required_columns]

In [22]:
column_names = [
    "Job Number", "Submit Time", "Wait Time", "Run Time", "Number of Allocated Nodes", "Average CPU Time Used",
    "Used Memory", "Requested Number of Nodes", "Requested Time", "Requested Memory", "Status", "User ID",
    "Group ID"]

swf.columns = column_names

In [23]:
swf['Executable Number'] = -1
swf['Queue Number'] = -1
swf['Partition Number'] = -1
swf['Preceding Job Number'] = -1
swf['Think Time from Preceding Job'] = -1

## Data Splitting

In [24]:
# Sort the dataframe by 'Submit Time'
swf = swf.sort_values(by='Submit Time', ascending=True)

# Calculate split index for 70-30 split
split_index = int(len(df) * 0.7)

# Split the dataframe
swf_train = swf.iloc[:split_index]
swf_test = swf.iloc[split_index:]

In [25]:
swf_train.shape

(155764, 18)

In [26]:
swf_test.shape

(66756, 18)

In [27]:
swf_test

,Job Number,Submit Time,Wait Time,Run Time,Number of Allocated Nodes,Average CPU Time Used,Used Memory,Requested Number of Nodes,Requested Time,Requested Memory,Status,User ID,Group ID,Executable Number,Queue Number,Partition Number,Preceding Job Number,Think Time from Preceding Job
177250,2272439,11480812,909,4465,1,4465,118,1,14400,118,1,1711,25200,-1,-1,-1,-1,-1
177274,4698474,11480812,1,4534,1,4534,118,1,14400,118,1,1711,25200,-1,-1,-1,-1,-1
177666,3661430,11480812,687,4387,1,4387,118,1,14400,118,1,1711,25200,-1,-1,-1,-1,-1
178002,949708,11480812,447,4594,1,4594,118,1,14400,118,1,1711,25200,-1,-1,-1,-1,-1
178135,2264709,11480812,1,4409,1,4409,118,1,14400,118,1,1711,25200,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218553,3214055,13832570,122,4,4,4,29,4,300,29,1,8,25200,-1,-1,-1,-1,-1
218614,558069,13833191,123,2,4,2,29,4,300,29,1,8,25200,-1,-1,-1,-1,-1
218556,1997479,13833813,122,3,4,3,29,4,300,29,1,8,25200,-1,-1,-1,-1,-1
218588,4424415,13834310,122,3,4,3,29,4,300,29,1,8,25200,-1,-1,-1,-1,-1


In [28]:
max_jobs = max_jobs_in_window(swf_test, window_minutes=5)
print(max_jobs)

915


In [33]:
def analyze_dataset(swf_df, train_df, test_df, dataset_name, original_shape):
    print(f"=== {dataset_name} STATS ===")

    # 1. Total, Train, Test shapes
    total_jobs = len(swf_df)
    train_jobs = len(train_df)
    test_jobs = len(test_df)
    dropped_jobs = original_shape[0] - total_jobs

    print(f"Original Row Count: {original_shape[0]}")
    print(f"Dropped Jobs during cleaning: {dropped_jobs}")
    print(f"Total Cleaned Jobs: {total_jobs}")
    print(f"Train Jobs: {train_jobs} ({train_jobs/total_jobs*100:.1f}%)")
    print(f"Test Jobs: {test_jobs} ({test_jobs/total_jobs*100:.1f}%)")

    # 2. Unique Users
    unique_users = swf_df["User ID"].nunique()
    print(f"Unique Users: {unique_users}")

    # 3. Repeated / Duplicate Jobs (Same user, resources, and runtime)

    duplicate_groups = swf_df.groupby(
        [
            "User ID",
            "Requested Number of Nodes",
            "Requested Time",
            "Run Time",
        ]
    )

    total_duplicates = 0
    for name, group in duplicate_groups:
        if len(group) > 1:
            # The first job is the original, subsequent ones are duplicates
            total_duplicates += len(group) - 1

    dup_percentage = (total_duplicates / total_jobs) * 100
    print(f"Repeated/Duplicate Jobs: {total_duplicates} ({dup_percentage:.2f}%)")
    print("-" * 30)


analyze_dataset(swf, swf_train, swf_test, "PM100", (231238, 35))

=== PM100 STATS ===
Original Row Count: 231238
Dropped Jobs during cleaning: 8718
Total Cleaned Jobs: 222520
Train Jobs: 155764 (70.0%)
Test Jobs: 66756 (30.0%)
Unique Users: 473
Repeated/Duplicate Jobs: 128546 (57.77%)
------------------------------


## Final Data Preparation

In [26]:
initial_time = int(swf_test['Submit Time'].min())

# Calculate the difference in seconds:
swf_test['Submit Time'] = swf_test['Submit Time'] - initial_time

In [27]:
# The maximum number of submitted jobs in a five minutes window is roughly 900. 
# The system has 980 nodes, so it can handle this rate easily.
# To stress the system, we will consider a 10x decrease in job resources.

In [28]:
swf_test_filtered = swf_test[swf_test["Number of Allocated Nodes"]<196]
print(f"There are {len(swf_test_filtered)} out of {len(swf_test)} with less than 98 nodes allocated")

There are 66734 out of 66756 with less than 98 nodes allocated


In [29]:
swf_test_1800_1 = swf_test_filtered.sample(n=1800, random_state=13)
swf_test_1800_1 = swf_test_1800_1.reset_index().drop(columns=['index'])
swf_test_1800_2 = swf_test_filtered.sample(n=1800, random_state=19)
swf_test_1800_2 = swf_test_1800_2.reset_index().drop(columns=['index'])
swf_test_1800_3 = swf_test_filtered.sample(n=1800, random_state=27)
swf_test_1800_3 = swf_test_1800_3.reset_index().drop(columns=['index'])
swf_test_1800_4 = swf_test_filtered.sample(n=1800, random_state=45)
swf_test_1800_4 = swf_test_1800_4.reset_index().drop(columns=['index'])
swf_test_1800_5 = swf_test_filtered.sample(n=1800, random_state=69)
swf_test_1800_5 = swf_test_1800_5.reset_index().drop(columns=['index'])

In [30]:
# Considering five minutes of simulation with 30 jobs per second
swf_test_1800_1["Submit Time"] = np.repeat(np.arange(300), 6)
swf_test_1800_2["Submit Time"] = np.repeat(np.arange(300), 6)
swf_test_1800_3["Submit Time"] = np.repeat(np.arange(300), 6)
swf_test_1800_4["Submit Time"] = np.repeat(np.arange(300), 6)
swf_test_1800_5["Submit Time"] = np.repeat(np.arange(300), 6)

## File Creation

In [31]:
# swf_test.to_csv('workloads/pm100/pm100.swf', sep='\t', index=False, header=False)

In [32]:
swf_test_1800_1.to_csv('workloads/pm100/pm100_1800_1.swf', sep='\t', index=False, header=False)
swf_test_1800_2.to_csv('workloads/pm100/pm100_1800_2.swf', sep='\t', index=False, header=False)
swf_test_1800_3.to_csv('workloads/pm100/pm100_1800_3.swf', sep='\t', index=False, header=False)
swf_test_1800_4.to_csv('workloads/pm100/pm100_1800_4.swf', sep='\t', index=False, header=False)
swf_test_1800_5.to_csv('workloads/pm100/pm100_1800_5.swf', sep='\t', index=False, header=False)